In [1]:

!pip install git+https://github.com/Lightning-AI/litgpt.git

  Cloning https://github.com/Lightning-AI/litgpt.git to /tmp/pip-req-build-rhjidgfy
  Running command git clone --filter=blob:none --quiet https://github.com/Lightning-AI/litgpt.git /tmp/pip-req-build-rhjidgfy
  Resolved https://github.com/Lightning-AI/litgpt.git to commit d19df7aa0d9031b97c04db84db410bcb459de665
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# run this 2x make sure you see /content/hf_cache directory
# first time it installs and restarts session without completing this cell

!pip install -U datasets fsspec

import os
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache"

from datasets import load_dataset

dataset = load_dataset("mhenrichsen/alpaca_2k_test")
print(dataset)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

alpaca_2000.parquet:   0%|          | 0.00/1.76M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 2000
    })
})


In [ ]:
/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920

In [18]:
import torch
import litgpt
from litgpt.lora import GPT, merge_lora_weights
from litgpt.data import Alpaca2k # this isnt verified
import lightning as L




class LitLLM(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = GPT.from_name(
            name="Llama-3.1-8B",
            lora_r=32,
            lora_alpha=16,
            lora_dropout=0.05,
            lora_query=True,
            lora_key=False,
            lora_value=True,
        )
        litgpt.lora.mark_only_lora_as_trainable(self.model)

    def on_train_start(self):
        state_dict = torch.load("/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920/pytorch_model.bin", mmap=True)
        self.model.load_state_dict(state_dict, strict=False)

    def training_step(self, batch):
        input_ids, targets = batch["input_ids"], batch["labels"]
        logits = self.model(input_ids)
        loss = litgpt.utils.chunked_cross_entropy(logits[..., :-1, :], targets[..., 1:])
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        warmup_steps = 10
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=0.0002, weight_decay=0.0, betas=(0.9, 0.95))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: step / warmup_steps)
        return [optimizer], [scheduler]


if __name__ == "__main__":
    data = Alpaca2k()
    tokenizer = litgpt.Tokenizer("/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920")
    data.connect(tokenizer, batch_size=1, max_seq_length=512)

    trainer = L.Trainer(
        devices=1,
        max_epochs=2,
        accumulate_grad_batches=8,
        precision="bf16-true",
    )
    with trainer.init_module(empty_init=True):
        model = LitLLM()
    print(f"data:{data}")
    print(f"model:{model}")

    trainer.fit(model, data)

    # Save final checkpoint
    merge_lora_weights(model.model)
    #stored in /content/drive/MyDrive/meta-llama/Meta-Llama-3-8B/finetuned.ckpt
    trainer.save_checkpoint("meta-llama/Meta-Llama-3-8B/finetuned.ckpt", weights_only=True)


INFO: Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.


data:{Train dataloader: None}
{Validation dataloader: size=NA}
{Test dataloader: None}
{Predict dataloader: None}
model:LitLLM(
  (model): GPT(
    (lm_head): LoRALinear(
      (lora_dropout): Dropout(p=0.05, inplace=False)
      (linear): Linear(in_features=4096, out_features=128256, bias=False)
    )
    (transformer): ModuleDict(
      (wte): Embedding(128256, 4096)
      (h): ModuleList(
        (0-31): 32 x Block(
          (norm_1): RMSNorm()
          (attn): CausalSelfAttention(
            (qkv): LoRAQKVLinear(
              (lora_dropout): Dropout(p=0.05, inplace=False)
              (linear): Linear(in_features=4096, out_features=6144, bias=False)
            )
            (proj): LoRALinear(
              (lora_dropout): Dropout(p=0.05, inplace=False)
              (linear): Linear(in_features=4096, out_features=4096, bias=False)
            )
          )
          (post_attention_norm): Identity()
          (norm_2): RMSNorm()
          (mlp): LLaMAMLP(
            (fc_1):

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name  | Type | Params | Mode 
---------------------------------------
0 | model | GPT  | 8.0 B  | train
---------------------------------------
13.6 M    Trainable params
8.0 B     Non-trainable params
8.0 B     Total params
32,175.571Total estimated model params size (MB)
712       Modules in train mode
0         Modules in eval mode
INFO:lightning.pytorch.callbacks.model_summary:
  | Name  | Type | Params | Mode 
---------------------------------------
0 | model | GPT  | 8.0 B  | train
---------------------------------------
13.6 M    Trainable params
8.0 B     Non-trainable params
8.0 B     Total params
32,175.571Total estimated model params size (MB)
712       Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=2` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=2` reached.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive

/content/drive/MyDrive


In [5]:
!pip install transformers accelerate

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
# device memory use cpu not cuda
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
# copied from /root/.cache to /content/drive/MyDrive....

#model = AutoModelForCausalLM.from_pretrained("/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920", device_map="auto")
#tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [9]:
from accelerate import init_empty_weights, infer_auto_device_map, load_checkpoint_and_dispatch
from transformers import AutoConfig, AutoModelForCausalLM

model_name = "/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920"

config = AutoConfig.from_pretrained(model_name)

with init_empty_weights():
    model = AutoModelForCausalLM.from_config(config)

device_map = infer_auto_device_map(model, max_memory={0: "13GiB", "cpu": "30GiB"}, no_split_module_classes=["LlamaDecoderLayer"])

model = load_checkpoint_and_dispatch(
    model,
    checkpoint=model_name,
    device_map=device_map,
    offload_folder="./offload"
)

  0%|          | 0/82 [00:00<?, ?w/s]

  0%|          | 0/104 [00:00<?, ?w/s]

  0%|          | 0/100 [00:00<?, ?w/s]

  0%|          | 0/5 [00:00<?, ?w/s]

In [16]:
!pip install safetensors

In [17]:
import torch
from safetensors.torch import load_file
import os

# Path to the directory containing model shards
shard_dir = "/content/drive/MyDrive/huggingface_model_downloads/meta-llama/models--meta-llama--Meta-Llama-3-8B/snapshots/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920"  # 🔁 Replace with your actual path

# List all .safetensors shard files
safetensor_files = sorted([
    os.path.join(shard_dir, f) for f in os.listdir(shard_dir)
    if f.endswith(".safetensors")
])

print(f"Found {len(safetensor_files)} safetensor shards.")

# Merge all shard dictionaries
full_state_dict = {}
for file in safetensor_files:
    shard = load_file(file)
    full_state_dict.update(shard)

# Save the merged state_dict as pytorch_model.bin
torch.save(full_state_dict, os.path.join(shard_dir, "pytorch_model.bin"))
print("Saved combined model to pytorch_model.bin")

Found 4 safetensor shards.
Saved combined model to pytorch_model.bin
